# Tier-1 SARG Q-Only Generation 150 Subject-Balanced (psxog/SARG, no metrics)

**Model:** `psxog/SARG` / `Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf` (4.92GB)

**Dataset:** `known_dataset_eval_subject_balanced_150.jsonl` 150 items (subject-balanced: Math 20, Science 20, Social 20, EVS 20, Bio 15, Geo 15, Econ 14, Chem 13, Phys 13)

**Inference:** `n_ctx=4096, n_gpu_layers=-1, n_batch=512, temperature=0.0, max_tokens=1024, top_p=1.0, build_messages question-only no system no context (question as user only)`

**Output:** `/content/sft_generations_qonly_subject_balanced_150.jsonl` (150 lines, `id, task_type, grade, subject, reference_answer, generation`)


## 1. Setup - Install & GPU Check

In [ ]:
!nvidia-smi
!pip install -q --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 llama-cpp-python
!pip install -q huggingface_hub
# Fallback if cu121 404:
# !pip install -q --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 llama-cpp-python
!python -c "import llama_cpp; print('GPU offload:', llama_cpp.llama_supports_gpu_offload())"
import sys
print("Python", sys.version)


Wed Sep  2 12:06:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Download Model + Dataset (150 Subject-Balanced)

In [ ]:
from huggingface_hub import hf_hub_download
import os, json

sft_repo, sft_file = "psxog/SARG", "Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf"
print(f"Downloading SFT {sft_repo}/{sft_file} ...")
sft_path = hf_hub_download(repo_id=sft_repo, filename=sft_file)
print(f"SFT: {sft_path} ({os.path.getsize(sft_path)/1e9:.2f} GB)")

dataset_path = "/content/known_dataset_eval_subject_balanced_150.jsonl"
if not os.path.exists(dataset_path):
    print(f"WARNING: {dataset_path} not found. Upload known_dataset_eval_subject_balanced_150.jsonl to /content")
    print("In Colab: from google.colab import files; files.upload() or drive.mount")
else:
    with open(dataset_path) as f:
        lines = f.readlines()
    print(f"Dataset: {len(lines)} items")
    print(json.loads(lines[0]).keys())
    print(f"Sample id: {json.loads(lines[0])['id']}")


Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf: reconstructing file:   0%|          |  0.00B / 4.92GB            

Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf: downloading bytes:           |  0.00B            

SFT: /root/.cache/huggingface/hub/models--psxog--SARG/snapshots/4966ecfa6f3a94b83ba11f57aa40d008fb1df13c/Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf (4.92 GB)
Dataset: 150 items
dict_keys(['id', 'context', 'question', 'reference_answer', 'grade', 'subject', 'task_type'])
Sample id: eval_001


## 3. Helpers - Load / Generate (identical, question-only)

In [ ]:
import json, os, re, time, gc
from llama_cpp import Llama

def build_messages(context, question):
    # Question-only: no system prompt, no context (context ignored, minimal diff)
    return [{"role": "user", "content": question}]

def load_model(model_path):
    llm = Llama(model_path=model_path, n_ctx=4096, n_gpu_layers=-1, n_batch=512, verbose=False)
    print(f"Loaded {os.path.basename(model_path)} n_ctx=4096 n_gpu_layers=-1")
    return llm

def generate_for_model(model_path, output_path):
    llm = load_model(model_path)
    # quick smoke test
    with open(dataset_path) as f:
        sample = json.loads(next(iter(f)))
    msgs = build_messages(sample["context"], sample["question"])
    out = llm.create_chat_completion(messages=msgs, temperature=0.0, max_tokens=50)
    print(f"Smoke {sample['id']}: {out['choices'][0]['message']['content'][:120]}...")
    !nvidia-smi --query-gpu=memory.used --format=csv,noheader
    start = time.time()
    with open(dataset_path) as fin, open(output_path, "w") as fout:
        for idx, line in enumerate(fin):
            item = json.loads(line)
            msgs = build_messages(item.get("context",""), item["question"])
            resp = llm.create_chat_completion(messages=msgs, temperature=0.0, max_tokens=1024, top_p=1.0)
            gen = resp["choices"][0]["message"]["content"].strip()
            rec = {"id": item["id"], "task_type": item.get("task_type",""), "grade": item.get("grade",""), "subject": item.get("subject",""), "reference_answer": item["reference_answer"], "generation": gen}
            fout.write(json.dumps(rec)+"\n")
            if (idx+1)%10==0:
                print(f"{idx+1}/150 elapsed {(time.time()-start)/60:.1f} min")
    print(f"Done {output_path} in {(time.time()-start)/60:.1f} min")
    !wc -l {output_path}
    return llm

def unload_model(llm):
    print("Unloading model...")
    try:
        llm.close()
    except Exception as e:
        print(f"close() warning: {e}")
    del llm
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except:
        pass
    time.sleep(2)
    !nvidia-smi --query-gpu=memory.used --format=csv,noheader
    print("VRAM should be ~1100 MiB baseline")


## 4. Generate 150 (qonly, no context)

In [ ]:
sft_out = "/content/sft_generations_qonly_subject_balanced_150.jsonl"
print("=== SFT/SARG psxog/SARG Question-Only 150 ===")
llm = generate_for_model(sft_path, sft_out)
unload_model(llm)
print("Generation saved")
!ls -lh /content/*150.jsonl


=== SFT/SARG psxog/SARG Question-Only 150 ===
Loaded Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf n_ctx=4096 n_gpu_layers=-1
Smoke eval_001: Lesson Plan: Introduction to Algebraic Expressions (Grade 6 Mathematics)

Duration: 40 minutes
Classroom Setting: Shared...
5353 MiB
10/150 elapsed 1.7 min
20/150 elapsed 3.8 min
30/150 elapsed 6.1 min
40/150 elapsed 8.3 min
50/150 elapsed 9.9 min
60/150 elapsed 12.3 min
70/150 elapsed 15.0 min
80/150 elapsed 17.5 min
90/150 elapsed 19.3 min
100/150 elapsed 22.0 min
110/150 elapsed 24.5 min
120/150 elapsed 26.6 min
130/150 elapsed 29.2 min
140/150 elapsed 31.0 min
150/150 elapsed 33.7 min
Done /content/sft_generations_qonly_subject_balanced_150.jsonl in 33.7 min
150 /content/sft_generations_qonly_subject_balanced_150.jsonl
Unloading model...
123 MiB
VRAM should be ~1100 MiB baseline
Generation saved
-rw-r--r-- 1 root root 311K Sep  2 12:06 /content/known_dataset_eval_subject_balanced_150.jsonl
-rw-r--r-- 1 root root 426K Sep  2 12:43 /content/sft_genera

## 5. Verify (no metrics, just file check)

In [ ]:
import json, os
print(f"Lines: {open(sft_out).read().count(chr(10))}")
first = json.loads(open(sft_out).readline())
print("Keys:", list(first.keys()))
print("First id:", first["id"], "gen len:", len(first["generation"]))
print("Unique ids:", len(set(json.loads(l)["id"] for l in open(sft_out))))
print("Empty gens:", sum(1 for l in open(sft_out) if not json.loads(l)["generation"].strip()))
print(json.dumps(first, indent=2)[:1200])
print(f"Ready: {sft_out} ({os.path.getsize(sft_out)/1e6:.2f} MB)")
# from google.colab import files; files.download(sft_out)


Lines: 150
Keys: ['id', 'task_type', 'grade', 'subject', 'reference_answer', 'generation']
First id: eval_001 gen len: 2018
Unique ids: 150
Empty gens: 0
{
  "id": "eval_001",
  "task_type": "lesson_plan",
  "grade": 6,
  "subject": "Mathematics",
  "reference_answer": "Lesson Plan: Introduction to Algebraic Expressions (Grade 6 Mathematics)\n\nDuration: 40 Minutes\nClassroom Context: Semi-rural plains school in Uttar Dinajpur, single-grade classroom, shared textbooks, no projector.\n\nLearning Objectives:\n1. Students will understand the concept of variables and constants using local agricultural produce and daily wage contexts.\n2. Students will be able to write simple algebraic expressions from word problems involving jute bundles, maize sacks, and daily earnings.\n\nTiming Segments:\n\n1. Introduction (5 Minutes):\n- The teacher draws local items on the blackboard, such as jute bundles harvested during the monsoon season.\n- Ask students: 'If Subodh helps pack several bundles of ju